# Cell 1 — Environment and reproducibility

In [1]:
import os, re, json, time, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd

import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

print("Python:", os.sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.9.0+cu126
CUDA available: True


# Cell 2 — Install and import dependencies 

In [2]:
import os, sys, subprocess, json
from pathlib import Path

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip"] + args)

pip(["-q", "install", "--no-cache-dir", "-U", "--force-reinstall", "--no-deps",
     "albucore==0.0.15",
     "albumentations==1.4.14"])

pip(["-q", "install", "--no-cache-dir", "-U", "--force-reinstall", "--no-deps",
     "segmentation-models-pytorch==0.3.3",
     "timm==0.9.12",
     "pretrainedmodels==0.7.4",
     "efficientnet-pytorch==0.7.1"])


import numpy as np
import cv2
import scipy
import skimage


import albumentations as A
import albucore
from albucore.utils import preserve_channel_dim
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("skimage:", skimage.__version__)
print("cv2:", cv2.__version__)
print("albumentations:", A.__version__)
print("albucore:", albucore.__version__)
print("smp:", smp.__version__)
print("preserve_channel_dim: OK")


OUT = Path("/kaggle/working/output")
OUT.mkdir(parents=True, exist_ok=True)
env = {
    "python": sys.version,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "skimage": skimage.__version__,
    "opencv": cv2.__version__,
    "albumentations": A.__version__,
    "albucore": albucore.__version__,
    "segmentation_models_pytorch": smp.__version__,
    "timm": __import__("timm").__version__,
    "pretrainedmodels": __import__("pretrainedmodels").__version__,
    "efficientnet_pytorch": __import__("efficientnet_pytorch").__version__,
}
(OUT / "env_versions.json").write_text(json.dumps(env, indent=2))
print("Saved:", OUT / "env_versions.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 42.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 321.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 182.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.7 MB/s eta 0:00:00
numpy: 2.0.2
scipy: 1.16.3
skimage: 0.25.2
cv2: 4.12.0
albumentations: 1.4.14
albucore: 0.0.15
smp: 0.3.3
preserve_channel_dim: OK
Saved: /kaggle/working/output/env_versions.json


In [3]:
import cv2, numpy as np, torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

print("cv2:", cv2.__version__)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("albumentations:", A.__version__)
print("smp:", smp.__version__)
print("Sanity imports: OK")

cv2: 4.12.0
numpy: 2.0.2
torch: 2.9.0+cu126
albumentations: 1.4.14
smp: 0.3.3
Sanity imports: OK


# Cell 3 — Protocol v1 config

In [4]:
# Cell 3 — Protocol v1 config (updated: speed + reproducibility controls)

import os, json, time, random, shutil
from pathlib import Path
import numpy as np
import torch

# Core protocol (match Model-1)
SEED = 7
IMG_SIZE = 448
BATCH_SIZE = 12
LR = 2e-4
WEIGHT_DECAY = 1e-4

EPOCHS = 40
EARLY_STOPPING_PATIENCE = 8

# Loss weights
W_BCE = 1.0
W_DICE = 1.0
W_EDGE = 0.5

# Performance knobs 
FAST_MODE = True          # speed-oriented cuDNN + channels_last
REPRO_MODE = False        # if True -> deterministic settings (slower)
USE_CHANNELS_LAST = True  # recommended with AMP for convnets
MATMUL_PRECISION = "high" # "high" is usually safe; official API
RESUME_IF_AVAILABLE = True

# DataLoader knobs (we will safely fallback to 0 if multiprocessing fails)
NUM_WORKERS = 2
PREFETCH_FACTOR = 2

MODEL_NAME = "deeplabv3plus_efficientnetb3"
RUN_NAME = "seg_deeplabv3p_proto1_448"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (DEVICE.type == "cuda")

# Output structure
OUT = Path("/kaggle/working/output")
RUN_DIR = OUT / "runs" / RUN_NAME
FIG_DIR = RUN_DIR / "figures"
CKPT_DIR = RUN_DIR / "checkpoints"
SAVED_MODELS_DIR = OUT / "SAVED_MODELS" / MODEL_NAME

for d in [OUT, RUN_DIR, FIG_DIR, CKPT_DIR, SAVED_MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BEST_PATH = CKPT_DIR / f"{MODEL_NAME}_best.pt"
LAST_PATH = CKPT_DIR / f"{MODEL_NAME}_last.pt"

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


if DEVICE.type == "cuda":
    if REPRO_MODE:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

print("DEVICE:", DEVICE, "| AMP:", AMP)
print("FAST_MODE:", FAST_MODE, "| REPRO_MODE:", REPRO_MODE)
print("RUN_DIR:", RUN_DIR)
print("CKPT_DIR:", CKPT_DIR)

DEVICE: cuda | AMP: True
FAST_MODE: True | REPRO_MODE: False
RUN_DIR: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448
CKPT_DIR: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints


# Cell 4 — Auto-discover Kaggle input paths

In [5]:
KAGGLE_INPUT = Path("/kaggle/input")

def find_first(pattern: str) -> Path:
    hits = sorted(KAGGLE_INPUT.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No match for pattern: {pattern}")
    return hits[0]

def find_dir_named(dir_name: str) -> Path:
    hits = sorted([p for p in KAGGLE_INPUT.rglob(dir_name) if p.is_dir()])
    if not hits:
        raise FileNotFoundError(f"Could not find directory named: {dir_name}")
    return hits[0]


seg_root = find_dir_named("crack_segmentation_dataset")
images_dir = seg_root / "images"
masks_dir  = seg_root / "masks"
assert images_dir.exists() and masks_dir.exists(), f"Missing images/masks under: {seg_root}"


train_split_csv = find_first("**/train_split.csv")
val_split_csv   = find_first("**/val_split.csv")

print("Seg root:", seg_root)
print("Images dir:", images_dir)
print("Masks dir:", masks_dir)
print("train_split.csv:", train_split_csv)
print("val_split.csv:", val_split_csv)

Seg root: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset
Images dir: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset/images
Masks dir: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset/masks
train_split.csv: /kaggle/input/datasets/arman00p/phase-3-csv-files/train_split.csv
val_split.csv: /kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv


# Cell 5 — Load split CSVs and resolve image/mask paths robustly

In [6]:
def build_stem_index(folder: Path):
    idx = {}
    for p in folder.glob("*"):
        if p.is_file():
            idx[p.stem] = p
    return idx

img_index = build_stem_index(images_dir)
msk_index = build_stem_index(masks_dir)

def resolve_paths(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalize column names if needed
    if "image_path" not in df.columns and "image" in df.columns:
        df["image_path"] = df["image"]
    if "mask_path" not in df.columns and "mask" in df.columns:
        df["mask_path"] = df["mask"]


    if "stem" not in df.columns:

        if "image_path" in df.columns:
            df["stem"] = df["image_path"].apply(lambda x: Path(str(x)).stem)
        else:
            raise ValueError("Split CSV must contain 'stem' or 'image_path'.")

    def _resolve_img(stem, p):
        p = Path(str(p)) if pd.notna(p) else None
        if p is not None and str(p).startswith("/kaggle") and p.exists():
            return str(p)
        if stem in img_index:
            return str(img_index[stem])
        raise FileNotFoundError(f"Could not resolve image for stem={stem}")

    def _resolve_msk(stem, p):
        p = Path(str(p)) if pd.notna(p) else None
        if p is not None and str(p).startswith("/kaggle") and p.exists():
            return str(p)
        if stem in msk_index:
            return str(msk_index[stem])
        raise FileNotFoundError(f"Could not resolve mask for stem={stem}")

    if "image_path" not in df.columns:
        df["image_path"] = np.nan
    if "mask_path" not in df.columns:
        df["mask_path"] = np.nan

    df["image_path"] = [
        _resolve_img(st, ip) for st, ip in zip(df["stem"].tolist(), df["image_path"].tolist())
    ]
    df["mask_path"] = [
        _resolve_msk(st, mp) for st, mp in zip(df["stem"].tolist(), df["mask_path"].tolist())
    ]

    return df[["stem", "image_path", "mask_path"]]

df_train = pd.read_csv(train_split_csv)
df_val   = pd.read_csv(val_split_csv)

df_train = resolve_paths(df_train)
df_val   = resolve_paths(df_val)

print("Train:", df_train.shape, "| Val:", df_val.shape)
print(df_train.head(2))
print(df_val.head(2))

Train: (9603, 3) | Val: (1695, 3)
                                stem  \
0                         forest_083   
1  CRACK500_20160328_151728_1281_361   

                                          image_path  \
0  /kaggle/input/datasets/lakshaymiddha/crack-seg...   
1  /kaggle/input/datasets/lakshaymiddha/crack-seg...   

                                           mask_path  
0  /kaggle/input/datasets/lakshaymiddha/crack-seg...  
1  /kaggle/input/datasets/lakshaymiddha/crack-seg...  
                                stem  \
0  Volker_DSC01686_242_386_1564_1468   
1         GAPS384_train_1166_541_641   

                                          image_path  \
0  /kaggle/input/datasets/lakshaymiddha/crack-seg...   
1  /kaggle/input/datasets/lakshaymiddha/crack-seg...   

                                           mask_path  
0  /kaggle/input/datasets/lakshaymiddha/crack-seg...  
1  /kaggle/input/datasets/lakshaymiddha/crack-seg...  


# Cell 6 — Augmentations (match Model-1 style) and Dataset/DataLoader

In [7]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch

cv2.setNumThreads(0)  

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(transpose_mask=True),
])

val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(transpose_mask=True),
])

class CrackSegDataset(torch.utils.data.Dataset):
    def __init__(self, df, aug):
        self.df = df.reset_index(drop=True)
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        ip, mp = r["image_path"], r["mask_path"]

        img = cv2.imread(ip, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"cv2 failed to read image: {ip}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(f"cv2 failed to read mask: {mp}")
        m = (m > 127).astype("uint8")

        out = self.aug(image=img, mask=m)
        x = out["image"]
        y = out["mask"]
        if y.ndim == 2:
            y = y.unsqueeze(0)
        y = y.float()
        return x, y

train_ds = CrackSegDataset(df_train, train_aug)
val_ds   = CrackSegDataset(df_val, val_aug)

def make_loader(ds, batch_size, shuffle, num_workers):
    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=shuffle,
    )
    if num_workers > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return torch.utils.data.DataLoader(ds, **kwargs)


try:
    train_loader = make_loader(train_ds, BATCH_SIZE, True, NUM_WORKERS)
    val_loader   = make_loader(val_ds,   BATCH_SIZE, False, NUM_WORKERS)
    _ = next(iter(train_loader))
    print("DataLoader OK with num_workers =", NUM_WORKERS)
except Exception as e:
    print("DataLoader multiprocessing failed, falling back to num_workers=0")
    print("Reason:", repr(e))
    train_loader = make_loader(train_ds, BATCH_SIZE, True, 0)
    val_loader   = make_loader(val_ds,   BATCH_SIZE, False, 0)

xb, yb = next(iter(train_loader))
print("Batch x:", xb.shape, xb.dtype, "| y:", yb.shape, yb.dtype)

DataLoader OK with num_workers = 2
Batch x: torch.Size([12, 3, 448, 448]) torch.float32 | y: torch.Size([12, 1, 448, 448]) torch.float32


# Cell 7 — Model, loss functions, metrics, optimizer, AMP scaler, scheduler

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import segmentation_models_pytorch as smp

# Safety checks (means: run config cell before this)
assert "DEVICE" in globals(), "DEVICE is not defined. Run the Protocol/Config cell first."
assert "LR" in globals() and "WEIGHT_DECAY" in globals(), "LR/WEIGHT_DECAY not defined. Run config cell."
assert "W_BCE" in globals() and "W_DICE" in globals() and "W_EDGE" in globals(), "Loss weights not defined."
AMP = (DEVICE.type == "cuda")  # keep consistent

# Model: DeepLabV3Plus with EfficientNet-B3 encoder (SMP)
model = smp.DeepLabV3Plus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

bce = nn.BCEWithLogitsLoss()

def soft_dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    inter = (probs * targets).sum(dim=1)
    den = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * inter + eps) / (den + eps)
    return 1.0 - dice.mean()

# Sobel edge loss (thin boundary emphasis)
_sobel_x = torch.tensor([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
_sobel_y = torch.tensor([[-1, -2, -1],
                         [ 0,  0,  0],
                         [ 1,  2,  1]], dtype=torch.float32).view(1, 1, 3, 3)

def edge_loss(logits, targets):
    probs = torch.sigmoid(logits)
    sx = _sobel_x.to(probs.device)
    sy = _sobel_y.to(probs.device)

    px = F.conv2d(probs, sx, padding=1)
    py = F.conv2d(probs, sy, padding=1)
    tx = F.conv2d(targets, sx, padding=1)
    ty = F.conv2d(targets, sy, padding=1)

    pg = torch.sqrt(px * px + py * py + 1e-6)
    tg = torch.sqrt(tx * tx + ty * ty + 1e-6)
    return F.l1_loss(pg, tg)

def total_loss(logits, targets):
    loss_bce = bce(logits, targets)
    loss_dice = soft_dice_loss(logits, targets)
    loss_edge = edge_loss(logits, targets)
    return W_BCE * loss_bce + W_DICE * loss_dice + W_EDGE * loss_edge

@torch.no_grad()
def iou_from_logits(logits, targets, thr=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()
    inter = (pred * targets).sum(dim=(1, 2, 3))
    union = (pred + targets - pred * targets).sum(dim=(1, 2, 3))
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()

@torch.no_grad()
def pixel_acc_from_logits(logits, targets, thr=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()
    return (pred == targets).float().mean().item()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Reduce LR when val_loss plateaus (mode='min') — matches your Model-1 protocol
scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, threshold=1e-4, min_lr=1e-6
)

scaler = torch.amp.GradScaler(enabled=AMP)

print("Model params (M):", sum(p.numel() for p in model.parameters()) / 1e6)
print("Scheduler:", type(scheduler).__name__, "| mode=min on val_loss")

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b3-5fb5a3c3.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b3-5fb5a3c3.pth


100%|██████████| 47.1M/47.1M [00:00<00:00, 125MB/s]


Model params (M): 11.680185
Scheduler: ReduceLROnPlateau | mode=min on val_loss


# Cell 8 — Training loop with checkpointing and final packaging zip

In [9]:
import time, json, shutil
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm


torch.set_float32_matmul_precision(MATMUL_PRECISION)


if FAST_MODE and USE_CHANNELS_LAST and DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)

autocast_kwargs = dict(device_type=DEVICE.type, enabled=AMP)
if AMP:
    autocast_kwargs["dtype"] = torch.float16


start_epoch = 1
history = []
best_iou = -1.0
bad_epochs = 0

if RESUME_IF_AVAILABLE and Path(LAST_PATH).exists():
    ckpt = torch.load(LAST_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    history = ckpt.get("history", [])
    start_epoch = int(ckpt.get("epoch", 0)) + 1
    # Recompute best_iou so far from history (robust)
    if len(history) > 0 and "val_iou" in history[0]:
        best_iou = max([h["val_iou"] for h in history])
    print(f"Resuming from epoch {start_epoch}, best_iou so far = {best_iou:.4f}")

def run_one_epoch(loader, train: bool):
    model.train(train)
    loss_sum, iou_sum, acc_sum, n = 0.0, 0.0, 0.0, 0

    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x = x.to(DEVICE, dtype=torch.float32, non_blocking=True)
        y = y.to(DEVICE, dtype=torch.float32, non_blocking=True)

        if FAST_MODE and USE_CHANNELS_LAST and DEVICE.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(**autocast_kwargs):
                logits = model(x)
                loss = total_loss(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                with torch.autocast(**autocast_kwargs):
                    logits = model(x)
                    loss = total_loss(logits, y)

        bs = x.size(0)
        loss_sum += loss.item() * bs

        with torch.no_grad():
            iou_sum += iou_from_logits(logits, y) * bs
            acc_sum += pixel_acc_from_logits(logits, y) * bs

        n += bs
        pbar.set_postfix(loss=loss_sum/n, iou=iou_sum/n)

    return loss_sum/n, iou_sum/n, acc_sum/n

t_start = time.time()

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_iou, train_acc = run_one_epoch(train_loader, train=True)
    val_loss,   val_iou,   val_acc   = run_one_epoch(val_loader,   train=False)

    # Protocol v1: scheduler monitors val_loss
    scheduler.step(val_loss)

    row = {
        "epoch": epoch,
        "train_loss": float(train_loss), "val_loss": float(val_loss),
        "train_iou": float(train_iou),   "val_iou": float(val_iou),
        "train_acc": float(train_acc),   "val_acc": float(val_acc),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "sec": float(time.time() - t0),
    }
    history.append(row)

    print(f"Epoch {epoch:02d} | val_iou={val_iou:.4f} val_loss={val_loss:.4f} lr={row['lr']:.2e}")

    # Save last checkpoint every epoch 
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "config": {
            "SEED": SEED, "IMG_SIZE": IMG_SIZE, "BATCH_SIZE": BATCH_SIZE,
            "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY,
            "LOSS_WEIGHTS": {"W_BCE": W_BCE, "W_DICE": W_DICE, "W_EDGE": W_EDGE},
            "MODEL_NAME": MODEL_NAME, "RUN_NAME": RUN_NAME,
            "FAST_MODE": FAST_MODE, "REPRO_MODE": REPRO_MODE,
            "USE_CHANNELS_LAST": USE_CHANNELS_LAST,
            "MATMUL_PRECISION": MATMUL_PRECISION,
            "NUM_WORKERS": getattr(train_loader, "num_workers", None),
        }
    }, LAST_PATH)

    # Save best by val_iou
    if val_iou > best_iou:
        best_iou = val_iou
        bad_epochs = 0
        shutil.copy2(LAST_PATH, BEST_PATH)
        print("best updated:", Path(BEST_PATH).name)
    else:
        bad_epochs += 1
        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

# Save history + config snapshot
hist_df = pd.DataFrame(history)
hist_csv = Path(RUN_DIR) / f"{MODEL_NAME}_history.csv"
hist_df.to_csv(hist_csv, index=False)

cfg_json = Path(RUN_DIR) / f"{MODEL_NAME}_train_config.json"
cfg_json.write_text(json.dumps({
    "SEED": SEED, "IMG_SIZE": IMG_SIZE, "BATCH_SIZE": BATCH_SIZE,
    "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY,
    "EPOCHS": EPOCHS, "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
    "LOSS_WEIGHTS": {"W_BCE": W_BCE, "W_DICE": W_DICE, "W_EDGE": W_EDGE},
    "MODEL_NAME": MODEL_NAME, "RUN_NAME": RUN_NAME,
    "FAST_MODE": FAST_MODE, "REPRO_MODE": REPRO_MODE,
    "USE_CHANNELS_LAST": USE_CHANNELS_LAST,
    "MATMUL_PRECISION": MATMUL_PRECISION,
    "NUM_WORKERS": NUM_WORKERS,
}, indent=2))

print("Saved:", hist_csv)
print("Saved:", cfg_json)
print("Best checkpoint:", BEST_PATH, "| exists:", Path(BEST_PATH).exists())
print("Last checkpoint:", LAST_PATH, "| exists:", Path(LAST_PATH).exists())
print("Total training minutes:", round((time.time() - t_start)/60, 2))


stamp = time.strftime("%Y%m%d_%H%M%S")
pkg_dir = Path(SAVED_MODELS_DIR)
pkg_dir.mkdir(parents=True, exist_ok=True)

best_named = pkg_dir / f"{MODEL_NAME}_best_{stamp}.pt"
last_named = pkg_dir / f"{MODEL_NAME}_last_{stamp}.pt"

shutil.copy2(BEST_PATH, best_named)
shutil.copy2(LAST_PATH, last_named)
shutil.copy2(hist_csv, pkg_dir / hist_csv.name)
shutil.copy2(cfg_json, pkg_dir / cfg_json.name)

zip_base = Path(OUT) / f"SAVED_MODELS_{MODEL_NAME}_{RUN_NAME}_{stamp}"
shutil.make_archive(str(zip_base), "zip", root_dir=str(pkg_dir))
zip_path = Path(str(zip_base) + ".zip")

print("Pack dir:", pkg_dir)
print("ZIP:", zip_path, "| bytes:", zip_path.stat().st_size)

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Epoch 01 | val_iou=0.5347 val_loss=0.5639 lr=2.00e-04
best updated: deeplabv3plus_efficientnetb3_best.pt


Epoch 02 | val_iou=0.5561 val_loss=0.5391 lr=2.00e-04
best updated: deeplabv3plus_efficientnetb3_best.pt


Epoch 03 | val_iou=0.5791 val_loss=0.5251 lr=2.00e-04
best updated: deeplabv3plus_efficientnetb3_best.pt


Epoch 04 | val_iou=0.5778 val_loss=0.5191 lr=2.00e-04


Epoch 05 | val_iou=0.5921 val_loss=0.5105 lr=2.00e-04
best updated: deeplabv3plus_efficientnetb3_best.pt


... (30 outputs left out)

Epoch 36 | val_iou=0.6246 val_loss=0.4802 lr=5.00e-05


Epoch 37 | val_iou=0.6253 val_loss=0.4804 lr=5.00e-05


Epoch 38 | val_iou=0.6244 val_loss=0.4804 lr=2.50e-05


Epoch 39 | val_iou=0.6251 val_loss=0.4812 lr=2.50e-05
Early stopping.
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/deeplabv3plus_efficientnetb3_history.csv
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/deeplabv3plus_efficientnetb3_train_config.json
Best checkpoint: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints/deeplabv3plus_efficientnetb3_best.pt | exists: True
Last checkpoint: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints/deeplabv3plus_efficientnetb3_last.pt | exists: True
Total training minutes: 341.82
Pack dir: /kaggle/working/output/SAVED_MODELS/deeplabv3plus_efficientnetb3
ZIP: /kaggle/working/output/SAVED_MODELS_deeplabv3plus_efficientnetb3_seg_deeplabv3p_proto1_448_20260222_163104.zip | bytes: 252144881


# Cell 9 — Verify outputs

In [10]:
from pathlib import Path
import os

print("OUT:", OUT)
print("RUN_DIR:", RUN_DIR)
print("CKPT_DIR:", CKPT_DIR)
print("BEST_PATH:", BEST_PATH, "| exists:", Path(BEST_PATH).exists())
print("LAST_PATH:", LAST_PATH, "| exists:", Path(LAST_PATH).exists())

print("\nList RUN_DIR:")
for p in sorted(Path(RUN_DIR).glob("**/*")):
    if p.is_file():
        print(" -", p.relative_to(RUN_DIR), "|", p.stat().st_size, "bytes")

print("\nRaw ls checkpoints:")
!ls -lah "{CKPT_DIR}" || true

OUT: /kaggle/working/output
RUN_DIR: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448
CKPT_DIR: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints
BEST_PATH: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints/deeplabv3plus_efficientnetb3_best.pt | exists: True
LAST_PATH: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints/deeplabv3plus_efficientnetb3_last.pt | exists: True

List RUN_DIR:
 - checkpoints/deeplabv3plus_efficientnetb3_best.pt | 136374367 bytes
 - checkpoints/deeplabv3plus_efficientnetb3_last.pt | 136375327 bytes
 - deeplabv3plus_efficientnetb3_history.csv | 5585 bytes
 - deeplabv3plus_efficientnetb3_train_config.json | 438 bytes

Raw ls checkpoints:
total 261M
drwxr-xr-x 2 root root 4.0K Feb 22 10:59 .
drwxr-xr-x 4 root root 4.0K Feb 22 16:31 ..
-rw-r--r-- 1 root root 131M Feb 22 15:21 deeplabv3plus_efficientnetb3_best.pt
-rw-r--r-- 1 root root 131M Feb 22 16:31 deeplabv3plus_efficientnetb3_last.pt


# Cell 10 — Load best checkpoint for evaluation

In [11]:
import torch
from pathlib import Path

assert Path(BEST_PATH).exists(), f"BEST_PATH not found: {BEST_PATH}"
ckpt = torch.load(BEST_PATH, map_location=DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Loaded BEST checkpoint:", BEST_PATH)
print("Epoch in checkpoint:", ckpt.get("epoch", None))

Loaded BEST checkpoint: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/checkpoints/deeplabv3plus_efficientnetb3_best.pt
Epoch in checkpoint: 31


# Cell 11 — Curves (loss/IoU/accuracy) saved as high-DPI PNGs

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


hist_csv = Path(RUN_DIR) / f"{MODEL_NAME}_history.csv"
if hist_csv.exists():
    hist_df = pd.read_csv(hist_csv)
else:
    assert "history" in globals() and len(history) > 0, "No history found. Training cell must finish."
    hist_df = pd.DataFrame(history)

FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, path: Path, dpi=500):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print("Saved:", path)

# Loss curve
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
ax.plot(hist_df["epoch"], hist_df["val_loss"], label="val_loss")
ax.set_title("Loss vs Epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
save_fig(fig, FIG_DIR / f"{MODEL_NAME}_loss_curve.png")
plt.close(fig)

# IoU curve
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(hist_df["epoch"], hist_df["train_iou"], label="train_iou")
ax.plot(hist_df["epoch"], hist_df["val_iou"], label="val_iou")
ax.set_title("IoU vs Epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("IoU"); ax.legend()
save_fig(fig, FIG_DIR / f"{MODEL_NAME}_iou_curve.png")
plt.close(fig)

# Pixel accuracy curve
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(hist_df["epoch"], hist_df["train_acc"], label="train_acc")
ax.plot(hist_df["epoch"], hist_df["val_acc"], label="val_acc")
ax.set_title("Pixel Accuracy vs Epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend()
save_fig(fig, FIG_DIR / f"{MODEL_NAME}_pixel_acc_curve.png")
plt.close(fig)

print("Curves saved to:", FIG_DIR)

Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures/deeplabv3plus_efficientnetb3_loss_curve.png
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures/deeplabv3plus_efficientnetb3_iou_curve.png
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures/deeplabv3plus_efficientnetb3_pixel_acc_curve.png
Curves saved to: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures


# Cell 12 — Validation metrics CSV + summary report (handles empty masks correctly)

In [13]:
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from pathlib import Path

EVAL_THR = 0.50
MIN_COMPONENT_AREA = 30

def remove_small_components(mask01: np.ndarray, min_area: int = 30) -> np.ndarray:
    m = (mask01.astype(np.uint8) * 255)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(mask01, dtype=np.uint8)
    for k in range(1, n):
        area = stats[k, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == k] = 1
    return out

class ValDatasetWithPaths(Dataset):
    def __init__(self, df, aug):
        self.df = df.reset_index(drop=True)
        self.aug = aug
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        ip, mp, stem = r["image_path"], r["mask_path"], r["stem"]
        img = cv2.imread(ip, cv2.IMREAD_COLOR); 
        if img is None: raise FileNotFoundError(ip)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if m is None: raise FileNotFoundError(mp)
        m = (m > 127).astype(np.uint8)
        out = val_aug(image=img, mask=m)
        x = out["image"]
        y = out["mask"]
        if y.ndim == 2: y = y.unsqueeze(0)
        return x, y.float(), stem, ip, mp

val_eval_ds = ValDatasetWithPaths(df_val, val_aug)
val_eval_loader = DataLoader(val_eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

rows = []
eps = 1e-6

AMP_EVAL = (DEVICE.type == "cuda")
autocast_kwargs = dict(device_type=DEVICE.type, enabled=AMP_EVAL)
if AMP_EVAL:
    autocast_kwargs["dtype"] = torch.float16

model.eval()
with torch.no_grad():
    for xb, yb, stems, ips, mps in tqdm(val_eval_loader, desc="Val metrics", leave=False):
        xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
        yb = yb.to(DEVICE, dtype=torch.float32, non_blocking=True)

        with torch.autocast(**autocast_kwargs):
            logits = model(xb)
        probs = torch.sigmoid(logits).float()
        pred = (probs > EVAL_THR).float()

        inter = (pred * yb).sum(dim=(1,2,3))
        union = (pred + yb - pred*yb).sum(dim=(1,2,3))
        iou = ((inter + eps) / (union + eps)).detach().cpu().numpy()

        dice = ((2*inter + eps) / (pred.sum(dim=(1,2,3)) + yb.sum(dim=(1,2,3)) + eps)).detach().cpu().numpy()

        gt_px = yb.sum(dim=(1,2,3)).detach().cpu().numpy()
        pr_px = pred.sum(dim=(1,2,3)).detach().cpu().numpy()

        probs_np = probs[:,0].detach().cpu().numpy()
        pred_np  = pred[:,0].detach().cpu().numpy().astype(np.uint8)

        # has_crack decision uses component filtering to avoid tiny noise blobs
        pred_has = []
        for b in range(pred_np.shape[0]):
            pm = remove_small_components(pred_np[b], min_area=MIN_COMPONENT_AREA)
            pred_has.append(int(pm.sum() > 0))
        pred_has = np.array(pred_has, dtype=int)

        gt_has = (gt_px > 0).astype(int)

        conf_p95 = np.percentile(probs_np.reshape(probs_np.shape[0], -1), 95, axis=1)
        conf_max = probs_np.reshape(probs_np.shape[0], -1).max(axis=1)

        conf_mean_pred = []
        for b in range(pred_np.shape[0]):
            m = pred_np[b].astype(bool)
            conf_mean_pred.append(float(probs_np[b][m].mean()) if m.any() else 0.0)

        for k in range(len(stems)):
            rows.append({
                "stem": str(stems[k]),
                "image_path": str(ips[k]),
                "mask_path": str(mps[k]),
                "iou": float(iou[k]),
                "dice": float(dice[k]),
                "gt_crack_pixels": int(gt_px[k]),
                "pred_crack_pixels": int(pr_px[k]),
                "gt_has_crack": int(gt_has[k]),
                "pred_has_crack": int(pred_has[k]),
                "conf_p95": float(conf_p95[k]),
                "conf_max": float(conf_max[k]),
                "conf_mean_pred": float(conf_mean_pred[k]),
            })

val_metrics_df = pd.DataFrame(rows)

# empty-aware IoU 
val_metrics_df["iou_emptyaware"] = val_metrics_df["iou"]
empty_gt = (val_metrics_df["gt_has_crack"] == 0)
empty_pr = (val_metrics_df["pred_has_crack"] == 0)
val_metrics_df.loc[empty_gt & empty_pr, "iou_emptyaware"] = 1.0
val_metrics_df.loc[empty_gt & (~empty_pr), "iou_emptyaware"] = 0.0

crack_df = val_metrics_df[val_metrics_df["gt_has_crack"] == 1]
noc_df   = val_metrics_df[val_metrics_df["gt_has_crack"] == 0]

summary = {
    "MODEL_NAME": MODEL_NAME,
    "RUN_NAME": RUN_NAME,
    "IMG_SIZE": int(IMG_SIZE),
    "BATCH_SIZE": int(BATCH_SIZE),
    "SEED": int(SEED),
    "EVAL_THR": float(EVAL_THR),
    "MIN_COMPONENT_AREA": int(MIN_COMPONENT_AREA),

    "n_val": int(len(val_metrics_df)),
    "n_gt_crack": int(len(crack_df)),
    "n_gt_nocrack": int(len(noc_df)),

    "mean_iou_all": float(val_metrics_df["iou"].mean()),
    "mean_iou_crack_only": float(crack_df["iou"].mean()) if len(crack_df) else float("nan"),
    "median_iou_crack_only": float(crack_df["iou"].median()) if len(crack_df) else float("nan"),
    "mean_dice_crack_only": float(crack_df["dice"].mean()) if len(crack_df) else float("nan"),

    "no_crack_false_positive_rate": float((noc_df["pred_has_crack"] == 1).mean()) if len(noc_df) else float("nan"),
    "no_crack_false_positive_count": int((noc_df["pred_has_crack"] == 1).sum()) if len(noc_df) else 0,

    "mean_iou_emptyaware_all": float(val_metrics_df["iou_emptyaware"].mean()),
}

val_metrics_path = Path(RUN_DIR) / f"{MODEL_NAME}_val_metrics.csv"
val_summary_path = Path(RUN_DIR) / f"{MODEL_NAME}_val_summary_report.csv"
val_metrics_df.to_csv(val_metrics_path, index=False)
pd.DataFrame([summary]).to_csv(val_summary_path, index=False)

print("Saved:", val_metrics_path)
print("Saved:", val_summary_path)
pd.DataFrame([summary]).T

Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/deeplabv3plus_efficientnetb3_val_metrics.csv
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/deeplabv3plus_efficientnetb3_val_summary_report.csv


,0
MODEL_NAME,deeplabv3plus_efficientnetb3
RUN_NAME,seg_deeplabv3p_proto1_448
IMG_SIZE,448
BATCH_SIZE,12
SEED,7
EVAL_THR,0.5
MIN_COMPONENT_AREA,30
n_val,1695
n_gt_crack,1477
n_gt_nocrack,218


# Cell 13 — Pixel PR curve + AUC (sampled pixels, memory-safe)

In [14]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_curve, auc
from tqdm import tqdm
from pathlib import Path

MAX_PR_PIXELS = 300_000
rng = np.random.default_rng(SEED)

probs_all = []
gts_all = []
collected = 0

model.eval()
with torch.no_grad():
    for xb, yb, *_ in tqdm(val_eval_loader, desc="PR sampling", leave=False):
        if collected >= MAX_PR_PIXELS:
            break
        xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
        yb = yb.to(DEVICE, dtype=torch.float32, non_blocking=True)

        with torch.autocast(**autocast_kwargs):
            logits = model(xb)
        probs = torch.sigmoid(logits).float()

        p = probs[:,0].detach().cpu().numpy().reshape(-1)
        g = yb[:,0].detach().cpu().numpy().reshape(-1).astype(np.uint8)

        take = min(g.size, MAX_PR_PIXELS - collected)
        idx = rng.choice(g.size, size=take, replace=False)
        probs_all.append(p[idx]); gts_all.append(g[idx])
        collected += take

probs_all = np.concatenate(probs_all)
gts_all = np.concatenate(gts_all)

prec, rec, _ = precision_recall_curve(gts_all, probs_all)
pr_auc = auc(rec, prec)

# Save curve image + AUC csv
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.8, 5.2))
ax.plot(rec, prec)
ax.set_title(f"Pixel Precision–Recall (AUC={pr_auc:.3f}, sampled={collected:,})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
fig.savefig(Path(FIG_DIR) / f"{MODEL_NAME}_pixel_pr_curve.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

auc_path = Path(RUN_DIR) / f"{MODEL_NAME}_val_pr_auc_pixel.csv"
pd.DataFrame([{"MODEL_NAME": MODEL_NAME, "pixel_pr_auc": float(pr_auc), "sampled_pixels": int(collected)}]).to_csv(auc_path, index=False)

print("Saved:", Path(FIG_DIR) / f"{MODEL_NAME}_pixel_pr_curve.png")
print("Saved:", auc_path)
print("Pixel PR AUC:", pr_auc)

Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures/deeplabv3plus_efficientnetb3_pixel_pr_curve.png
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/deeplabv3plus_efficientnetb3_val_pr_auc_pixel.csv
Pixel PR AUC: 0.8795991124577368


# Cell 14 — IoU histograms, crack area hist, confidence hist

In [15]:
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = Path(FIG_DIR)

# IoU all
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.hist(val_metrics_df["iou"], bins=30)
ax.set_title("Validation IoU Distribution (All)")
ax.set_xlabel("IoU"); ax.set_ylabel("Count")
fig.savefig(FIG_DIR / f"{MODEL_NAME}_val_iou_hist_all.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

# IoU crack-only
crack_df = val_metrics_df[val_metrics_df["gt_has_crack"] == 1]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.hist(crack_df["iou"], bins=30)
ax.set_title("Validation IoU Distribution (Crack-present only)")
ax.set_xlabel("IoU"); ax.set_ylabel("Count")
fig.savefig(FIG_DIR / f"{MODEL_NAME}_val_iou_hist_crack_only.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

# GT crack area
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.hist(crack_df["gt_crack_pixels"], bins=30)
ax.set_title("GT Crack Area Proxy (pixels) — Crack-present only")
ax.set_xlabel("GT crack pixels"); ax.set_ylabel("Count")
fig.savefig(FIG_DIR / f"{MODEL_NAME}_gt_crack_area_hist.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

# Pred crack area
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.hist(val_metrics_df["pred_crack_pixels"], bins=30)
ax.set_title("Predicted Crack Area Proxy (pixels) — All val")
ax.set_xlabel("Pred crack pixels"); ax.set_ylabel("Count")
fig.savefig(FIG_DIR / f"{MODEL_NAME}_pred_crack_area_hist.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

# Confidence hist
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.hist(val_metrics_df["conf_p95"], bins=30)
ax.set_title("Confidence (P95 pixel prob) — All val")
ax.set_xlabel("conf_p95"); ax.set_ylabel("Count")
fig.savefig(FIG_DIR / f"{MODEL_NAME}_confidence_p95_hist.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.hist(val_metrics_df["conf_mean_pred"], bins=30)
ax.set_title("Confidence (mean prob inside predicted mask) — All val")
ax.set_xlabel("conf_mean_pred"); ax.set_ylabel("Count")
fig.savefig(FIG_DIR / f"{MODEL_NAME}_confidence_mean_pred_hist.png", dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved histogram PNGs to:", FIG_DIR)

Saved histogram PNGs to: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures


# Cell 15 — Top-K worst cases gallery (crack-present only)

In [16]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from pathlib import Path

K = 12
worst = (val_metrics_df[val_metrics_df["gt_has_crack"] == 1]
         .sort_values("iou", ascending=True)
         .head(K)
         .reset_index(drop=True))

def read_rgb(path, size):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img

def read_mask01(path, size):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    m = (m > 127).astype(np.uint8)
    m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
    return m

def overlay_mask(img_rgb, mask01, alpha=0.35):
    out = img_rgb.astype(np.float32).copy()
    m = mask01.astype(bool)
    out[m] = (1-alpha)*out[m] + alpha*np.array([255,0,0], dtype=np.float32)
    return out.clip(0,255).astype(np.uint8)

fig, axes = plt.subplots(len(worst), 5, figsize=(16, 3.0 * len(worst)))
fig.suptitle("Top-K Worst (Crack-present only): Image | GT | Pred | Overlay | Prob heatmap", y=1.002)

model.eval()
with torch.no_grad():
    for r in range(len(worst)):
        ip = worst.loc[r, "image_path"]
        mp = worst.loc[r, "mask_path"]
        stem = worst.loc[r, "stem"]

        img = read_rgb(ip, IMG_SIZE)
        gt  = read_mask01(mp, IMG_SIZE)

        out = val_aug(image=img, mask=gt)
        x = out["image"].unsqueeze(0).to(DEVICE, dtype=torch.float32)

        with torch.autocast(**autocast_kwargs):
            logits = model(x)
        prob = torch.sigmoid(logits).squeeze().float().cpu().numpy()
        pred = (prob > EVAL_THR).astype(np.uint8)

        axes[r,0].imshow(img); axes[r,0].axis("off"); axes[r,0].set_title(f"{stem}", fontsize=10)
        axes[r,1].imshow(gt, cmap="gray"); axes[r,1].axis("off"); axes[r,1].set_title("GT", fontsize=10)
        axes[r,2].imshow(pred, cmap="gray"); axes[r,2].axis("off"); axes[r,2].set_title(f"Pred IoU={worst.loc[r,'iou']:.3f}", fontsize=10)
        axes[r,3].imshow(overlay_mask(img, pred)); axes[r,3].axis("off"); axes[r,3].set_title("Overlay", fontsize=10)
        axes[r,4].imshow(prob, cmap="magma"); axes[r,4].axis("off"); axes[r,4].set_title("Prob", fontsize=10)

out_path = Path(FIG_DIR) / f"{MODEL_NAME}_topk_worst_crack_only_gallery.png"
fig.savefig(out_path, dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Saved:", out_path)

Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures/deeplabv3plus_efficientnetb3_topk_worst_crack_only_gallery.png


# Cell 16 — Mask → AABB and OBB overlays + JSONL detections

In [17]:
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from pathlib import Path

def mask_to_aabbs(mask01, min_area=30):
    m = (mask01.astype(np.uint8) * 255)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    boxes = []
    for k in range(1, n):
        x, y, w, h, area = stats[k]
        if area < min_area: 
            continue
        boxes.append({"x1": int(x), "y1": int(y), "x2": int(x+w), "y2": int(y+h), "area_px": int(area)})
    boxes.sort(key=lambda d: d["area_px"], reverse=True)
    return boxes

def mask_to_obbs(mask01, min_area=30):
    m = (mask01.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    obbs = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < float(min_area):
            continue
        rect = cv2.minAreaRect(cnt)
        box = cv2.boxPoints(rect).astype(np.int32).tolist()
        obbs.append({"box": box, "area_px": int(area), "rect": {"cx": rect[0][0], "cy": rect[0][1], "w": rect[1][0], "h": rect[1][1], "angle": rect[2]}})
    obbs.sort(key=lambda d: d["area_px"], reverse=True)
    return obbs

def draw_aabbs(img_rgb, boxes, color=(255,0,0), lw=2, topk=3):
    out = img_rgb.copy()
    for b in boxes[:topk]:
        cv2.rectangle(out, (b["x1"], b["y1"]), (b["x2"], b["y2"]), color, lw)
    return out

def draw_obbs(img_rgb, obbs, color=(255,0,0), lw=2, topk=3):
    out = img_rgb.copy()
    for o in obbs[:topk]:
        pts = np.array(o["box"], dtype=np.int32).reshape(-1,1,2)
        cv2.polylines(out, [pts], True, color, lw)
    return out

def remove_small(mask01, min_area=30):
    return remove_small_components(mask01, min_area=min_area)

N_SHOW = 10
idxs = np.random.default_rng(SEED).choice(len(df_val), size=min(N_SHOW, len(df_val)), replace=False)

detections_path = Path(RUN_DIR) / f"{MODEL_NAME}_val_detections_from_masks.jsonl"
fig, axes = plt.subplots(len(idxs), 4, figsize=(16, 3.1*len(idxs)))
fig.suptitle("Mask→Detection outputs: Image | GT AABB+OBB | Pred AABB+OBB | Pred overlay", y=1.002)

with detections_path.open("w") as f, torch.no_grad():
    for r, idx in enumerate(idxs):
        row = df_val.iloc[int(idx)]
        ip, mp, stem = row["image_path"], row["mask_path"], row["stem"]

        img = read_rgb(ip, IMG_SIZE)
        gt = read_mask01(mp, IMG_SIZE)

        out = val_aug(image=img, mask=gt)
        x = out["image"].unsqueeze(0).to(DEVICE, dtype=torch.float32)

        with torch.autocast(**autocast_kwargs):
            logits = model(x)
        prob = torch.sigmoid(logits).squeeze().float().cpu().numpy()
        pred = (prob > EVAL_THR).astype(np.uint8)

        gt_det = remove_small(gt, MIN_COMPONENT_AREA)
        pr_det = remove_small(pred, MIN_COMPONENT_AREA)

        gt_aabb = mask_to_aabbs(gt_det, MIN_COMPONENT_AREA)
        pr_aabb = mask_to_aabbs(pr_det, MIN_COMPONENT_AREA)
        gt_obb  = mask_to_obbs(gt_det, MIN_COMPONENT_AREA)
        pr_obb  = mask_to_obbs(pr_det, MIN_COMPONENT_AREA)

        record = {
            "stem": stem,
            "image_path": str(ip),
            "mask_path": str(mp),
            "thr": float(EVAL_THR),
            "min_component_area": int(MIN_COMPONENT_AREA),
            "pred_area_px": int(pr_det.sum()),
            "conf_p95": float(np.percentile(prob.reshape(-1), 95)),
            "aabb_pred": pr_aabb,
            "obb_pred": pr_obb,
            "aabb_gt": gt_aabb,
            "obb_gt": gt_obb,
        }
        f.write(json.dumps(record) + "\n")

        gt_vis = draw_obbs(draw_aabbs(img, gt_aabb, color=(0,255,0)), gt_obb, color=(0,255,0))
        pr_vis = draw_obbs(draw_aabbs(img, pr_aabb, color=(255,0,0)), pr_obb, color=(255,0,0))
        ov_vis = overlay_mask(img, pr_det, alpha=0.35)

        axes[r,0].imshow(img); axes[r,0].axis("off"); axes[r,0].set_title(stem, fontsize=10)
        axes[r,1].imshow(gt_vis); axes[r,1].axis("off"); axes[r,1].set_title("GT", fontsize=10)
        axes[r,2].imshow(pr_vis); axes[r,2].axis("off"); axes[r,2].set_title("Pred", fontsize=10)
        axes[r,3].imshow(ov_vis); axes[r,3].axis("off"); axes[r,3].set_title("Overlay", fontsize=10)

over_path = Path(FIG_DIR) / f"{MODEL_NAME}_bbox_obb_overlays.png"
fig.savefig(over_path, dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved:", detections_path)
print("Saved:", over_path)

Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/deeplabv3plus_efficientnetb3_val_detections_from_masks.jsonl
Saved: /kaggle/working/output/runs/seg_deeplabv3p_proto1_448/figures/deeplabv3plus_efficientnetb3_bbox_obb_overlays.png


# Cell 17 — Package everything into one ZIP (checkpoints + CSVs + figures + JSONL)

In [18]:
import time, shutil, json
from pathlib import Path

stamp = time.strftime("%Y%m%d_%H%M%S")

bundle_dir = Path(OUT) / "BUNDLES" / f"{MODEL_NAME}_{RUN_NAME}_{stamp}"
bundle_dir.mkdir(parents=True, exist_ok=True)


for p in [Path(BEST_PATH), Path(LAST_PATH)]:
    assert p.exists(), f"Missing checkpoint: {p}"
    shutil.copy2(p, bundle_dir / p.name)


for p in [
    Path(RUN_DIR) / f"{MODEL_NAME}_history.csv",
    Path(RUN_DIR) / f"{MODEL_NAME}_train_config.json",
    Path(RUN_DIR) / f"{MODEL_NAME}_val_metrics.csv",
    Path(RUN_DIR) / f"{MODEL_NAME}_val_summary_report.csv",
    Path(RUN_DIR) / f"{MODEL_NAME}_val_pr_auc_pixel.csv",
    Path(RUN_DIR) / f"{MODEL_NAME}_val_detections_from_masks.jsonl",
]:
    if p.exists():
        shutil.copy2(p, bundle_dir / p.name)


fig_src = Path(FIG_DIR)
if fig_src.exists():
    fig_dst = bundle_dir / "figures"
    fig_dst.mkdir(parents=True, exist_ok=True)
    for p in fig_src.glob("*.png"):
        shutil.copy2(p, fig_dst / p.name)


zip_base = Path(OUT) / f"BUNDLE_{MODEL_NAME}_{RUN_NAME}_{stamp}"
shutil.make_archive(str(zip_base), "zip", root_dir=str(bundle_dir))
zip_path = Path(str(zip_base) + ".zip")

print("Bundle folder:", bundle_dir)
print("ZIP:", zip_path, "| bytes:", zip_path.stat().st_size)


from IPython.display import FileLink
FileLink(str(zip_path))

Bundle folder: /kaggle/working/output/BUNDLES/deeplabv3plus_efficientnetb3_seg_deeplabv3p_proto1_448_20260222_163426
ZIP: /kaggle/working/output/BUNDLE_deeplabv3plus_efficientnetb3_seg_deeplabv3p_proto1_448_20260222_163426.zip | bytes: 284141744


/kaggle/working/output/BUNDLE_deeplabv3plus_efficientnetb3_seg_deeplabv3p_proto1_448_20260222_163426.zip

# Cell 18 — Zip the entire /kaggle/working/output folder

In [19]:
import shutil, time
from pathlib import Path
from IPython.display import FileLink

stamp = time.strftime("%Y%m%d_%H%M%S")
src_dir = Path("/kaggle/working/output")
zip_base = Path("/kaggle/working") / f"ALL_OUTPUT_{stamp}"

shutil.make_archive(str(zip_base), "zip", root_dir=str(src_dir))
zip_path = Path(str(zip_base) + ".zip")

print("ZIP:", zip_path, "| bytes:", zip_path.stat().st_size)
FileLink(str(zip_path))

ZIP: /kaggle/working/ALL_OUTPUT_20260222_163452.zip | bytes: 1356871306


/kaggle/working/ALL_OUTPUT_20260222_163452.zip